# Step 3: Fine-tuning 기법 비교 실습

## 학습 목표
이 노트북을 완료하면 다음을 이해할 수 있습니다:
- **세 가지 Fine-tuning 기법**의 원리와 차이점
- 각 기법의 **장단점**과 **적합한 사용 시나리오**
- SageMaker Training Job을 활용한 **분산 학습**

## Fine-tuning 기법 비교

| 기법 | 학습 파라미터 | 학습 시간 | 메모리 | 적합한 경우 |
|------|--------------|----------|--------|------------|
| **Full Fine-tuning** | 전체 (4M) | 느림 | 많음 | 데이터 많고, 태스크가 많이 다를 때 |
| **Layer Freezing** | Classifier만 (~2K) | 빠름 | 적음 | 데이터 적고, 태스크가 비슷할 때 |
| **LoRA** | Adapter만 (~10K) | 빠름 | 적음 | 대규모 모델, 효율성 중요할 때 |

## 기법 상세 설명

### 1️⃣ Full Fine-tuning (전체 미세조정)
```
┌─────────────────┐
│    Backbone     │ ← 학습 O (gradient 흐름)
│  (EfficientNet) │
└────────┬────────┘
         ↓
┌─────────────────┐
│   Classifier    │ ← 학습 O
└─────────────────┘
```
- **모든 레이어**의 가중치를 업데이트
- 가장 높은 성능 가능, 하지만 **과적합 위험**
- 학습 데이터가 충분할 때 권장

### 2️⃣ Layer Freezing (레이어 동결)
```
┌─────────────────┐
│    Backbone     │ ← 동결 ❄️ (gradient 없음)
│  (EfficientNet) │
└────────┬────────┘
         ↓
┌─────────────────┐
│   Classifier    │ ← 학습 O
└─────────────────┘
```
- Backbone을 **동결**하고 Classifier만 학습
- 빠르고 **과적합 방지**, 하지만 성능 제한
- 학습 데이터가 적거나, 빠른 실험이 필요할 때

### 3️⃣ LoRA (Low-Rank Adaptation)
```
┌─────────────────┐
│    Backbone     │ ← 동결 ❄️
│  + LoRA Adapter │ ← 학습 O (작은 행렬만)
└────────┬────────┘
         ↓
┌─────────────────┐
│   Classifier    │ ← 학습 O
└─────────────────┘
```
- 원본 가중치는 동결, **작은 Adapter 행렬**만 학습
- 파라미터 효율적 (원본의 ~1% 미만)
- **LLM Fine-tuning**에서 특히 인기

## 3.1 환경 설정

In [ ]:
import json
import os
import sagemaker
from sagemaker.pytorch import PyTorch
from sagemaker.experiments.run import Run
from datetime import datetime
from pathlib import Path

# ============================================
# 프로젝트 경로 자동 설정
# ============================================
home_dir = Path.home()
PROJECT_ROOT = home_dir / 'deepfake-detection-sagemaker'
notebook_dir = PROJECT_ROOT / '3_fine_tuning'
os.chdir(notebook_dir)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Current Dir: {os.getcwd()}")

# 설정 로드
config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

# SageMaker 세션
sagemaker_session = sagemaker.Session()
role = config['role']
bucket = config['bucket']
prefix = config['prefix']

# Experiment 설정
EXPERIMENT_NAME = "deepfake-detection-kodf"
RUN_NAME = f"finetuning-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"Role: {role[:50]}...")
print(f"Bucket: {bucket}")
print(f"Experiment: {EXPERIMENT_NAME}")

## 3.2 Fine-tuning 기법 선택

실습에서 비교할 **세 가지 기법**을 선택합니다.

각 기법별로 SageMaker Training Job을 실행하고, 
이후 노트북에서 **성능을 비교**합니다.

> 💡 **실습 팁**: 시간이 부족하면 하나만 선택해서 실행해도 됩니다!

In [ ]:
# ============================================
# 🎯 Fine-tuning 기법 선택
# ============================================
# 비교할 기법들을 선택하세요 (True/False)

METHODS = {
    'full': True,           # 1️⃣ Full Fine-tuning (전체 학습)
    'freeze': True,         # 2️⃣ Layer Freezing (Backbone 동결)
    'lora': True,           # 3️⃣ LoRA (Low-Rank Adaptation)
}

# 공통 하이퍼파라미터
BASE_HYPERPARAMETERS = {
    'epochs': 5,              # 에포크 수 (시간 고려)
    'batch-size': 32,         # 배치 크기
    'learning-rate': 0.0001,  # 학습률
    'model-name': 'efficientnet_b0'
}

print("=" * 50)
print("📋 선택된 Fine-tuning 기법:")
print("=" * 50)
for method, enabled in METHODS.items():
    status = "✅ 실행" if enabled else "⬜ 건너뜀"
    print(f"  {method.upper():10} : {status}")
print("=" * 50)
print(f"\n총 {sum(METHODS.values())}개 기법 실행 예정")
print(f"예상 소요 시간: 약 {sum(METHODS.values()) * 15}분")

## 3.3 SageMaker Estimator 설정

각 Fine-tuning 기법별로 SageMaker Training Job을 실행합니다.

### SageMaker Training Job이란?
- AWS 관리형 인프라에서 학습 실행
- **GPU 인스턴스** 자동 프로비저닝
- **Spot Instance**로 비용 최대 70% 절감
- 학습 완료 후 모델을 **S3에 자동 저장**

### 인스턴스 타입
| 타입 | GPU | 메모리 | 시간당 비용 |
|------|-----|--------|------------|
| ml.g4dn.xlarge | T4 1개 | 16GB | ~$0.74 |
| ml.g4dn.2xlarge | T4 1개 | 32GB | ~$1.05 |
| ml.p3.2xlarge | V100 1개 | 16GB | ~$3.82 |

In [ ]:
# Spot Instance 사용 여부 (비용 ~70% 절감)
USE_SPOT = True

# 메트릭 정의 (CloudWatch에서 수집)
METRIC_DEFINITIONS = [
    {'Name': 'train:loss', 'Regex': r'Train Loss: ([0-9\.]+)'},
    {'Name': 'train:accuracy', 'Regex': r'Train Acc: ([0-9\.]+)%'},
    {'Name': 'val:loss', 'Regex': r'Val Loss: ([0-9\.]+)'},
    {'Name': 'val:accuracy', 'Regex': r'Val Acc: ([0-9\.]+)%'},
]

def create_estimator(method, hyperparameters):
    """기법별 SageMaker Estimator 생성"""
    
    # 기법별 하이퍼파라미터 추가
    hp = hyperparameters.copy()
    hp['finetune-method'] = method
    
    if method == 'lora':
        hp['lora-rank'] = 8  # LoRA rank
    
    estimator = PyTorch(
        entry_point='train.py',
        source_dir='.',
        role=role,
        instance_count=1,
        instance_type='ml.g4dn.xlarge',
        framework_version='2.0.0',
        py_version='py310',
        hyperparameters=hp,
        output_path=f's3://{bucket}/{prefix}/output/{method}',
        sagemaker_session=sagemaker_session,
        # Spot Instance 설정
        use_spot_instances=USE_SPOT,
        max_wait=7200 if USE_SPOT else None,
        max_run=3600,
        # Job 이름에 기법 포함
        base_job_name=f'deepfake-{method}',
        # 메트릭 정의 추가
        metric_definitions=METRIC_DEFINITIONS
    )
    
    return estimator

print("✅ Estimator 생성 함수 준비 완료")
print(f"   Spot Instance: {'활성화 (비용 ~70% 절감)' if USE_SPOT else '비활성화'}")
print(f"   Instance Type: ml.g4dn.xlarge (NVIDIA T4 GPU)")
print(f"   메트릭 수집: {len(METRIC_DEFINITIONS)}개 정의됨")

## 3.4 Training Job 실행

### 학습 과정 이해

각 기법별로 SageMaker Training Job이 실행됩니다:

1. **인스턴스 프로비저닝**: AWS가 GPU 인스턴스(ml.g4dn.xlarge)를 자동 할당
2. **데이터 다운로드**: S3에서 학습/검증 데이터를 인스턴스로 복사
3. **학습 실행**: train.py 스크립트가 실행되며 모델 학습
4. **모델 저장**: 학습 완료 후 model.tar.gz로 S3에 자동 업로드

### 로그 확인 포인트

학습 중 출력되는 로그에서 확인할 것:
- `학습 파라미터: X / Y (Z%)` - 기법별 학습 파라미터 비율
- `Train Loss`, `Train Acc` - 매 에포크 학습 손실/정확도
- `Val Loss`, `Val Acc` - 매 에포크 검증 손실/정확도
- `Best model saved!` - 최고 성능 모델 저장 시점

In [ ]:
# ============================================
# 🚀 선택된 기법들로 Training Job 실행
# ============================================

# 데이터 채널 설정
data_channels = {
    'train': config['s3_train_path'],
    'val': config['s3_val_path']
}

print("데이터 채널:")
for k, v in data_channels.items():
    print(f"  {k}: {v}")

# 각 기법별 결과 저장용 딕셔너리
training_results = {}

# 선택된 기법들만 실행
selected_methods = [m for m, enabled in METHODS.items() if enabled]

print(f"\n{'='*60}")
print(f"  총 {len(selected_methods)}개 기법 학습 시작")
print(f"{'='*60}")

for i, method in enumerate(selected_methods, 1):
    print(f"\n[{i}/{len(selected_methods)}] {method.upper()} Fine-tuning 시작...")
    print("-" * 50)
    
    # Estimator 생성
    estimator = create_estimator(method, BASE_HYPERPARAMETERS)
    
    # SageMaker Experiments와 함께 실행
    run_name = f"{method}-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    
    with Run(
        experiment_name=EXPERIMENT_NAME,
        run_name=run_name,
        sagemaker_session=sagemaker_session
    ) as run:
        # 하이퍼파라미터 로깅
        run.log_parameters(BASE_HYPERPARAMETERS)
        run.log_parameter("finetune_method", method)
        run.log_parameter("instance_type", "ml.g4dn.xlarge")
        run.log_parameter("use_spot", str(USE_SPOT))  # boolean을 문자열로 변환
        
        # Training 실행
        estimator.fit(data_channels, wait=True, logs='All')
        
        # 결과 저장
        run.log_parameter("model_data", estimator.model_data)
    
    # 결과 기록
    training_results[method] = {
        'model_data': estimator.model_data,
        'training_job_name': estimator.latest_training_job.name
    }
    
    print(f"✅ {method.upper()} 완료!")
    print(f"   모델 경로: {estimator.model_data}")

print(f"\n{'='*60}")
print(f"  모든 Training Job 완료!")
print(f"{'='*60}")

## 3.5 학습 결과 확인 및 저장

### 결과 해석 포인트

학습이 완료되면 각 기법별로 다음을 비교해보세요:

| 확인 항목 | Full | Freeze | LoRA |
|----------|------|--------|------|
| 학습 파라미터 수 | ~4M (100%) | ~2K (0.05%) | ~10K (0.25%) |
| 최종 Train Acc | 높음 | 중간 | 중간~높음 |
| 최종 Val Acc | 높음 | 중간 | 중간~높음 |
| 과적합 위험 | 높음 | 낮음 | 낮음 |

> 💡 **관찰**: Full Fine-tuning은 Train Acc는 높지만 Val Acc와 차이가 크면 과적합!

In [ ]:
# ============================================
# 📊 학습 결과 요약 및 config 저장
# ============================================

print("=" * 60)
print("  📊 Fine-tuning 결과 요약")
print("=" * 60)

for method, result in training_results.items():
    print(f"\n🔹 {method.upper()}")
    print(f"   모델 경로: {result['model_data']}")
    print(f"   Job 이름: {result['training_job_name']}")

# config.json 업데이트 (모든 기법의 모델 경로 저장)
config['training_results'] = training_results

# 기본 모델 경로 (첫 번째 기법)
if training_results:
    first_method = list(training_results.keys())[0]
    config['model_data'] = training_results[first_method]['model_data']
    config['training_job_name'] = training_results[first_method]['training_job_name']

config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"\n✅ 설정 저장 완료: {config_path}")
print(f"   저장된 기법: {list(training_results.keys())}")

In [ ]:
# ============================================
# 📈 SageMaker Experiments에서 학습 메트릭 확인
# ============================================
from sagemaker.analytics import TrainingJobAnalytics

print("각 기법별 학습 메트릭 조회 중...\n")

for method, result in training_results.items():
    print(f"🔹 {method.upper()} 메트릭:")
    print("-" * 40)
    try:
        analytics = TrainingJobAnalytics(result['training_job_name'])
        df = analytics.dataframe()
        if not df.empty:
            display(df.tail())
        else:
            print("  (메트릭 데이터 없음)")
    except Exception as e:
        print(f"  조회 실패: {e}")
    print()

## 완료!

SageMaker Fine-tuning이 완료되었습니다.

### 학습 결과 요약

| 기법 | 특징 | 예상 결과 |
|------|------|----------|
| **Full** | 전체 파라미터 학습 | 높은 정확도, 과적합 가능성 |
| **Freeze** | Classifier만 학습 | 빠른 학습, 안정적 |
| **LoRA** | Adapter 행렬만 학습 | 효율적, LLM에서 인기 |

### 다음 단계에서 확인할 것

1. **각 기법별 테스트 정확도** 비교
2. **학습 파라미터 수 대비 성능** 분석
3. **어떤 기법이 우리 태스크에 적합한지** 판단

**➡️ 다음 단계: `4_after_evaluation/evaluate_after.ipynb`**

각 Fine-tuning 기법의 성능을 테스트 데이터로 평가합니다!